In [1]:
import sys
print(sys.version)



3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]


In [2]:
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
CUDA available: True


In [7]:
import os
import sqlite3
from modules.preprocess import TextPreprocessor

# Initialize NLP preprocessor
tp = TextPreprocessor()

# Database path (under rag/)
DB_PATH = "../rag/sql_store.db"
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("✅ Connected to SQLite:", DB_PATH)


✅ Connected to SQLite: ../rag/sql_store.db


In [8]:
# Drop existing table (if any) and recreate
cursor.execute("DROP TABLE IF EXISTS documents;")

cursor.execute("""
CREATE VIRTUAL TABLE documents
USING fts5(
    id UNINDEXED,
    title,
    content,
    entities,
    tokenize='porter'  -- enables stemming for better keyword matching
);
""")

conn.commit()
print("✅ FTS5 table created with porter tokenizer")


✅ FTS5 table created with porter tokenizer


In [9]:
sample_docs = [
    ("1", "HybridLM Overview", 
     "HybridLM is a hybrid model combining symbolic and semantic reasoning. Built by OpenAI in 2025."),
    
    ("2", "MedicalGPT Ethics", 
     "MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024."),
    
    ("3", "Finance AI", 
     "FinAI predicts S&P 500 market trends using transformer models."),
    
    ("4", "LLaMA Evolution", 
     "LLaMA 3 outperformed GPT-4 in efficiency.")
]

for doc_id, title, content in sample_docs:
    processed = tp.clean_text(content)
    cursor.execute(
        "INSERT INTO documents (id, title, content, entities) VALUES (?, ?, ?, ?)", 
        (doc_id, title, processed, "")
    )

conn.commit()
print(f"✅ Inserted {len(sample_docs)} sample documents")


✅ Inserted 4 sample documents


In [12]:
def symbolic_search(query: str, top_k: int = 3):
    """
    Performs a symbolic keyword/entity-aware search over the SQLite FTS5 index.
    Uses OR-based term expansion to ensure partial matches return results.
    Skips placeholder tokens like <num> or <url> that break MATCH syntax.
    """
    q_clean = tp.clean_text(query)
    terms = q_clean.split()

    # Filter out any placeholders or invalid tokens (like <num>, <url>)
    safe_terms = [t for t in terms if not any(ch in t for ch in "<>")]

    if not safe_terms:
        print("⚠️ No valid search tokens in query.")
        return []

    match_expr = " OR ".join(safe_terms)

    cursor.execute("""
        SELECT title, content, rank
        FROM documents
        WHERE documents MATCH ?
        ORDER BY rank
        LIMIT ?;
    """, (match_expr, top_k))

    results = cursor.fetchall()
    return results


In [13]:
test_queries = [
    "Which AI model from Google DeepMind advanced medical NLP?",
    "What predicts S&P 500 trends using transformers?",
    "Who built HybridLM?",
    "Which model outperformed GPT-4?",
]

for q in test_queries:
    print(f"\n🔍 Query: {q}")
    results = symbolic_search(q)
    if not results:
        print("⚠️ No symbolic matches found.")
    else:
        for r in results:
            print(f"📘 {r[0]} → {r[1][:100]}...")



🔍 Query: Which AI model from Google DeepMind advanced medical NLP?
📘 MedicalGPT Ethics → MedicalGPT a system by Google DeepMind advanced medical nlp after 2024...
📘 Finance AI → FinAI predicts S P 500 market trends using transformer models...
📘 HybridLM Overview → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 2025...

🔍 Query: What predicts S&P 500 trends using transformers?
📘 Finance AI → FinAI predicts S P 500 market trends using transformer models...

🔍 Query: Who built HybridLM?
📘 HybridLM Overview → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 2025...

🔍 Query: Which model outperformed GPT-4?
📘 LLaMA Evolution → LLaMA <num> outperformed GPT <Num> in efficiency...
📘 Finance AI → FinAI predicts S P 500 market trends using transformer models...
📘 HybridLM Overview → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 2025...


In [14]:
!pip install faiss-cpu sentence-transformers --quiet


In [15]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.


In [16]:
# Initialize semantic model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_text(texts):
    """Encode list of texts into dense vectors."""
    embeddings = embed_model.encode(texts, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    return np.array(embeddings)


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [17]:
# Fetch all documents from FTS5
cursor.execute("SELECT id, title, content FROM documents;")
docs = cursor.fetchall()

doc_ids, titles, contents = zip(*docs)
embeddings = embed_text(contents)

# Create FAISS index (L2 normalized vectors → cosine similarity)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"✅ FAISS index built with {len(docs)} documents | dim={embeddings.shape[1]}")


✅ FAISS index built with 4 documents | dim=384


In [18]:
def semantic_search(query: str, top_k: int = 3):
    """
    Semantic search using FAISS over SentenceTransformer embeddings.
    Returns ranked (title, content, score) tuples.
    """
    q_clean = tp.clean_text(query)
    q_vec = embed_text([q_clean])
    scores, indices = index.search(q_vec, top_k)
    
    results = []
    for idx, score in zip(indices[0], scores[0]):
        title = titles[idx]
        content = contents[idx]
        results.append((title, content, float(score)))
    return results


In [19]:
test_queries = [
    "Which AI model from Google DeepMind advanced medical NLP?",
    "What predicts S&P 500 trends using transformers?",
    "Who built HybridLM?",
    "Which model outperformed GPT-4?",
]

for q in test_queries:
    print(f"\n🧠 Query: {q}")
    results = semantic_search(q)
    for title, content, score in results:
        print(f"📘 {title} ({score:.3f}) → {content[:100]}...")



🧠 Query: Which AI model from Google DeepMind advanced medical NLP?
📘 MedicalGPT Ethics (0.814) → MedicalGPT a system by Google DeepMind advanced medical nlp after 2024...
📘 HybridLM Overview (0.449) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 2025...
📘 LLaMA Evolution (0.109) → LLaMA <num> outperformed GPT <Num> in efficiency...

🧠 Query: What predicts S&P 500 trends using transformers?
📘 Finance AI (0.704) → FinAI predicts S P 500 market trends using transformer models...
📘 LLaMA Evolution (0.183) → LLaMA <num> outperformed GPT <Num> in efficiency...
📘 MedicalGPT Ethics (0.104) → MedicalGPT a system by Google DeepMind advanced medical nlp after 2024...

🧠 Query: Who built HybridLM?
📘 HybridLM Overview (0.333) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 2025...
📘 LLaMA Evolution (0.175) → LLaMA <num> outperformed GPT <Num> in efficiency...
📘 MedicalGPT Ethics (0.017) → MedicalGPT a system by Go

In [20]:
import sqlite3

conn = sqlite3.connect("symbolic_rag.db")
cursor = conn.cursor()

for row in cursor.execute("SELECT title FROM documents;"):
    print(row)

conn.close()


OperationalError: no such table: documents

In [21]:
import sqlite3

# Connect (creates file if not exists)
conn = sqlite3.connect("symbolic_rag.db")
cursor = conn.cursor()

# 1️⃣ Create FTS5 table
cursor.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS documents USING fts5(
    title, 
    content
);
""")

# 2️⃣ Insert sample docs
docs = [
    ("MedicalGPT Ethics", "MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024."),
    ("Finance AI", "FinAI predicts S&P 500 market trends using transformer models."),
    ("HybridLM Overview", "HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 2025."),
    ("LLaMA Evolution", "LLaMA 3 outperformed GPT-4 in efficiency."),
]

cursor.executemany("INSERT INTO documents (title, content) VALUES (?, ?)", docs)

# 3️⃣ Commit changes
conn.commit()

# 4️⃣ Verify contents
for row in cursor.execute("SELECT title, content FROM documents;"):
    print(f"📘 {row[0]} → {row[1][:60]}...")

conn.close()


📘 MedicalGPT Ethics → MedicalGPT, a system by Google DeepMind, advanced medical NL...
📘 Finance AI → FinAI predicts S&P 500 market trends using transformer model...
📘 HybridLM Overview → HybridLM is a hybrid model combining symbolic and semantic re...
📘 LLaMA Evolution → LLaMA 3 outperformed GPT-4 in efficiency....


In [22]:
import sqlite3

def symbolic_search(query, top_k=3):
    """Simple symbolic keyword-based search using SQLite FTS5"""
    conn = sqlite3.connect("symbolic_rag.db")
    cursor = conn.cursor()

    # Clean and prepare query
    q = query.replace("<", "").replace(">", "")
    terms = q.split()
    match_expr = " OR ".join(terms)

    cursor.execute(f"""
        SELECT title, content 
        FROM documents 
        WHERE documents MATCH ? 
        LIMIT ?;
    """, (match_expr, top_k))

    results = cursor.fetchall()
    conn.close()
    return results


# Test symbolic retrieval
test_queries = [
    "Which AI model from Google DeepMind advanced medical NLP?",
    "What predicts S&P 500 trends using transformers?",
    "Who built HybridLM?",
    "Which model outperformed GPT-4?"
]

for q in test_queries:
    print(f"\n🔍 Query: {q}")
    results = symbolic_search(q)
    if not results:
        print("⚠️ No symbolic matches found.")
    for r in results:
        print(f"📘 {r[0]} → {r[1][:80]}...")



🔍 Query: Which AI model from Google DeepMind advanced medical NLP?


OperationalError: fts5: syntax error near "?"

In [24]:
import sqlite3
import re

def symbolic_search(query, top_k=3):
    """Full-text keyword search using SQLite FTS5 (sanitized safe version)."""
    conn = sqlite3.connect("symbolic_rag.db")
    cursor = conn.cursor()

    # Step 1: Sanitize query (remove punctuation that breaks FTS5)
    q = re.sub(r'[^\w\s]', ' ', query)  # keep only letters, numbers, and spaces
    q = q.replace("<", "").replace(">", "").replace('"', "")
    q = re.sub(r'\s+', ' ', q).strip()

    # Step 2: Build FTS5 expression
    terms = q.split()
    match_expr = " OR ".join(terms)

    # Step 3: Execute query safely
    sql = f"""
        SELECT title, content
        FROM documents
        WHERE documents MATCH "{match_expr}"
        LIMIT {top_k};
    """

    try:
        results = cursor.execute(sql).fetchall()
    except sqlite3.OperationalError as e:
        print(f"⚠️ SQLite FTS error for query: {match_expr}\n→ {e}")
        results = []

    conn.close()
    return results


# 🔍 Test queries again
test_queries = [
    "Which AI model from Google DeepMind advanced medical NLP?",
    "What predicts S&P 500 trends using transformers?",
    "Who built HybridLM?",
    "Which model outperformed GPT-4?"
]

for q in test_queries:
    print(f"\n🔍 Query: {q}")
    results = symbolic_search(q)
    if not results:
        print("⚠️ No symbolic matches found.")
    for r in results:
        print(f"📘 {r[0]} → {r[1][:80]}...")



🔍 Query: Which AI model from Google DeepMind advanced medical NLP?
📘 MedicalGPT Ethics → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....
📘 Finance AI → FinAI predicts S&P 500 market trends using transformer models....
📘 HybridLM Overview → HybridLM is a hybrid model combining symbolic and semantic reasoning built by Ope...

🔍 Query: What predicts S&P 500 trends using transformers?
📘 Finance AI → FinAI predicts S&P 500 market trends using transformer models....

🔍 Query: Who built HybridLM?
📘 HybridLM Overview → HybridLM is a hybrid model combining symbolic and semantic reasoning built by Ope...

🔍 Query: Which model outperformed GPT-4?
📘 HybridLM Overview → HybridLM is a hybrid model combining symbolic and semantic reasoning built by Ope...
📘 LLaMA Evolution → LLaMA 3 outperformed GPT-4 in efficiency....


In [25]:
from sentence_transformers import SentenceTransformer
import faiss


In [26]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import sqlite3

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 1: Load all docs from your SQLite DB
conn = sqlite3.connect("symbolic_rag.db")
cursor = conn.cursor()
cursor.execute("SELECT rowid, title, content FROM documents;")
docs = cursor.fetchall()
conn.close()

# Step 2: Create embeddings for content
texts = [d[2] for d in docs]
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.array(embeddings).astype("float32")

# Step 3: Build FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Step 4: Keep metadata
doc_map = {i: {"title": docs[i][1], "content": docs[i][2]} for i in range(len(docs))}

print(f"✅ FAISS index built with {len(docs)} documents and dimension {embeddings.shape[1]}")


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built with 4 documents and dimension 384


In [27]:
def semantic_search(query, top_k=3):
    q_emb = model.encode([query], normalize_embeddings=True)
    D, I = index.search(np.array(q_emb).astype("float32"), top_k)
    results = [(doc_map[i]["title"], doc_map[i]["content"], float(D[0][j])) for j, i in enumerate(I[0])]
    return results


In [28]:
queries = [
    "Google’s AI for medical text understanding",
    "AI predicting stock markets",
    "symbolic and semantic hybrid model by OpenAI"
]

for q in queries:
    print(f"\n🧠 Query: {q}")
    for title, content, score in semantic_search(q):
        print(f"📘 {title} ({score:.3f}) → {content[:80]}...")



🧠 Query: Google’s AI for medical text understanding


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📘 MedicalGPT Ethics (0.691) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....
📘 HybridLM Overview (0.337) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by Ope...
📘 Finance AI (0.060) → FinAI predicts S&P 500 market trends using transformer models....

🧠 Query: AI predicting stock markets


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📘 Finance AI (0.450) → FinAI predicts S&P 500 market trends using transformer models....
📘 MedicalGPT Ethics (0.189) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....
📘 HybridLM Overview (0.187) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by Ope...

🧠 Query: symbolic and semantic hybrid model by OpenAI


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📘 HybridLM Overview (0.767) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by Ope...
📘 MedicalGPT Ethics (0.361) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....
📘 Finance AI (0.139) → FinAI predicts S&P 500 market trends using transformer models....


In [29]:
import numpy as np
import re

def detect_query_type(query: str):
    """
    Heuristic scoring to decide α and β.
    - Entity-heavy (specific models/orgs): higher α
    - Abstract or conceptual: higher β
    """
    symbolic_keywords = ["GPT", "LLaMA", "HybridLM", "OpenAI", "DeepMind", "FinAI"]
    if any(word.lower() in query.lower() for word in symbolic_keywords):
        return 0.7, 0.3  # α=0.7, β=0.3 (favor symbolic)
    else:
        return 0.4, 0.6  # α=0.4, β=0.6 (favor semantic)


def hybrid_search(query, top_k=3):
    """Fuse symbolic and semantic retrieval with adaptive weighting."""
    α, β = detect_query_type(query)

    # --- symbolic retrieval ---
    sym_results = symbolic_search(query, top_k=top_k)
    sym_dict = {title: {"content": content, "score": α * (1 - i / top_k)} 
                for i, (title, content) in enumerate(sym_results)}

    # --- semantic retrieval ---
    sem_results = semantic_search(query, top_k=top_k)
    for title, content, score in sem_results:
        weighted_score = β * score
        if title in sym_dict:
            sym_dict[title]["score"] += weighted_score
        else:
            sym_dict[title] = {"content": content, "score": weighted_score}

    # --- rank fusion results ---
    fused = sorted(sym_dict.items(), key=lambda x: x[1]["score"], reverse=True)
    return fused[:top_k]


In [30]:
test_queries = [
    "Which AI model from Google DeepMind advanced medical NLP?",
    "AI predicting stock markets",
    "Who built HybridLM?",
    "Which model outperformed GPT-4?",
    "symbolic and semantic hybrid model by OpenAI"
]

for q in test_queries:
    print(f"\n🤖 Query: {q}")
    results = hybrid_search(q)
    for title, data in results:
        print(f"📘 {title} ({data['score']:.3f}) → {data['content'][:90]}...")



🤖 Query: Which AI model from Google DeepMind advanced medical NLP?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📘 MedicalGPT Ethics (0.937) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....
📘 Finance AI (0.467) → FinAI predicts S&P 500 market trends using transformer models....
📘 HybridLM Overview (0.373) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 202...

🤖 Query: AI predicting stock markets


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📘 Finance AI (0.670) → FinAI predicts S&P 500 market trends using transformer models....
📘 MedicalGPT Ethics (0.113) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....
📘 HybridLM Overview (0.112) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 202...

🤖 Query: Who built HybridLM?


...

📘 HybridLM Overview (0.814) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 202...
📘 LLaMA Evolution (0.042) → LLaMA 3 outperformed GPT-4 in efficiency....
📘 MedicalGPT Ethics (0.012) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....

🤖 Query: Which model outperformed GPT-4?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📘 HybridLM Overview (0.700) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 202...
📘 LLaMA Evolution (0.668) → LLaMA 3 outperformed GPT-4 in efficiency....
📘 MedicalGPT Ethics (0.079) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....

🤖 Query: symbolic and semantic hybrid model by OpenAI


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📘 MedicalGPT Ethics (0.808) → MedicalGPT, a system by Google DeepMind, advanced medical NLP after 2024....
📘 HybridLM Overview (0.697) → HybridLM is a hybrid model combining symbolic and semantic reasoning built by OpenAI in 202...
📘 Finance AI (0.042) → FinAI predicts S&P 500 market trends using transformer models....
